In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv(
    "../datasets/processed/dividend_puzzle_fundamental_dataset.csv"
)

print("Shape:", data.shape)
print(data.columns.tolist())

Shape: (110, 17)
['company', 'ticker', 'year', 'year_end_price', 'dividend_per_share', 'dividend_yield', 'dividend_growth', 'dividend_growth_category', 'annual_stock_return', 'return_category', 'total_return', 'return_volatility', 'average_return', 'dividend_consistency', 'revenue', 'net_income', 'total_assets']


In [2]:
fundamentals = pd.read_csv(
    "../datasets/raw/fundamentals_real.csv"
)

print("Fundamentals shape:", fundamentals.shape)
print(fundamentals.columns.tolist())

Fundamentals shape: (46, 5)
['ticker', 'year', 'revenue', 'net_income', 'total_assets']


In [3]:
data["ticker"] = (
    data["ticker"]
    .astype(str)
    .str.strip()
    .str.upper()
)

fundamentals["ticker"] = (
    fundamentals["ticker"]
    .astype(str)
    .str.strip()
    .str.upper()
)

data["year"] = pd.to_numeric(
    data["year"],
    errors="coerce"
)

fundamentals["year"] = pd.to_numeric(
    fundamentals["year"],
    errors="coerce"
)

print("Keys cleaned successfully.")

Keys cleaned successfully.


In [4]:
common = data.merge(
    fundamentals[["ticker", "year"]],
    on=["ticker", "year"],
    how="inner"
)

print("Matching company-year rows:", len(common))

display(
    common[
        ["ticker", "year"]
    ].sort_values(["ticker", "year"])
)

Matching company-year rows: 43


,ticker,year
0,AAPL,2021
1,AAPL,2022
2,AAPL,2023
3,AAPL,2024
4,AAPL,2025
5,JNJ,2021
6,JNJ,2022
7,JNJ,2023
8,JNJ,2024
9,JNJ,2025


In [5]:
data = data.merge(
    fundamentals[
        [
            "ticker",
            "year",
            "revenue",
            "net_income",
            "total_assets"
        ]
    ],
    on=["ticker", "year"],
    how="left"
)

print("Final shape after merge:", data.shape)

print(
    data.columns.tolist()
)

Final shape after merge: (110, 20)
['company', 'ticker', 'year', 'year_end_price', 'dividend_per_share', 'dividend_yield', 'dividend_growth', 'dividend_growth_category', 'annual_stock_return', 'return_category', 'total_return', 'return_volatility', 'average_return', 'dividend_consistency', 'revenue_x', 'net_income_x', 'total_assets_x', 'revenue_y', 'net_income_y', 'total_assets_y']


In [6]:
print(
    data[
        [
            "ticker",
            "year",
            "revenue",
            "net_income",
            "total_assets"
        ]
    ].head(20)
)

KeyError: "['revenue', 'net_income', 'total_assets'] not in index"

In [7]:
print("ALL COLUMNS:")
for i, col in enumerate(data.columns):
    print(i, repr(col))

ALL COLUMNS:
0 'company'
1 'ticker'
2 'year'
3 'year_end_price'
4 'dividend_per_share'
5 'dividend_yield'
6 'dividend_growth'
7 'dividend_growth_category'
8 'annual_stock_return'
9 'return_category'
10 'total_return'
11 'return_volatility'
12 'average_return'
13 'dividend_consistency'
14 'revenue_x'
15 'net_income_x'
16 'total_assets_x'
17 'revenue_y'
18 'net_income_y'
19 'total_assets_y'


In [8]:
# Use the newly merged fundamental values
data["revenue"] = data["revenue_y"]
data["net_income"] = data["net_income_y"]
data["total_assets"] = data["total_assets_y"]

print("Columns fixed successfully!")

print(
    data[
        ["ticker", "year", "revenue", "net_income", "total_assets"]
    ].head(20).to_string(index=False)
)

Columns fixed successfully!
ticker  year      revenue   net_income  total_assets
  AAPL  2015          NaN          NaN           NaN
  AAPL  2016          NaN          NaN           NaN
  AAPL  2017          NaN          NaN           NaN
  AAPL  2018          NaN          NaN           NaN
  AAPL  2019          NaN          NaN           NaN
  AAPL  2020          NaN          NaN           NaN
  AAPL  2021          NaN          NaN           NaN
  AAPL  2022 3.943280e+11 9.980300e+10  3.527550e+11
  AAPL  2023 3.832850e+11 9.699500e+10  3.525830e+11
  AAPL  2024 3.910350e+11 9.373600e+10  3.649800e+11
  AAPL  2025 4.161610e+11 1.120100e+11  3.592410e+11
   JNJ  2015          NaN          NaN           NaN
   JNJ  2016          NaN          NaN           NaN
   JNJ  2017          NaN          NaN           NaN
   JNJ  2018          NaN          NaN           NaN
   JNJ  2019          NaN          NaN           NaN
   JNJ  2020          NaN          NaN           NaN
   JNJ  2021      

In [9]:
data = data.drop(
    columns=[
        "revenue_x",
        "net_income_x",
        "total_assets_x",
        "revenue_y",
        "net_income_y",
        "total_assets_y"
    ]
)

print(data.columns.tolist())

['company', 'ticker', 'year', 'year_end_price', 'dividend_per_share', 'dividend_yield', 'dividend_growth', 'dividend_growth_category', 'annual_stock_return', 'return_category', 'total_return', 'return_volatility', 'average_return', 'dividend_consistency', 'revenue', 'net_income', 'total_assets']


In [10]:
data["profit_margin"] = np.where(
    data["revenue"] != 0,
    (data["net_income"] / data["revenue"]) * 100,
    np.nan
)

print("Profit Margin calculated.")

Profit Margin calculated.


In [11]:
data["roa"] = np.where(
    data["total_assets"] != 0,
    (data["net_income"] / data["total_assets"]) * 100,
    np.nan
)

print("ROA calculated.")

ROA calculated.


In [12]:
print(
    data[
        [
            "ticker",
            "year",
            "revenue",
            "net_income",
            "total_assets",
            "profit_margin",
            "roa"
        ]
    ].head(20).to_string(index=False)
)

ticker  year      revenue   net_income  total_assets  profit_margin       roa
  AAPL  2015          NaN          NaN           NaN            NaN       NaN
  AAPL  2016          NaN          NaN           NaN            NaN       NaN
  AAPL  2017          NaN          NaN           NaN            NaN       NaN
  AAPL  2018          NaN          NaN           NaN            NaN       NaN
  AAPL  2019          NaN          NaN           NaN            NaN       NaN
  AAPL  2020          NaN          NaN           NaN            NaN       NaN
  AAPL  2021          NaN          NaN           NaN            NaN       NaN
  AAPL  2022 3.943280e+11 9.980300e+10  3.527550e+11      25.309641 28.292441
  AAPL  2023 3.832850e+11 9.699500e+10  3.525830e+11      25.306234 27.509835
  AAPL  2024 3.910350e+11 9.373600e+10  3.649800e+11      23.971256 25.682503
  AAPL  2025 4.161610e+11 1.120100e+11  3.592410e+11      26.915064 31.179626
   JNJ  2015          NaN          NaN           NaN            

In [13]:
print(
    data[
        [
            "revenue",
            "net_income",
            "total_assets",
            "profit_margin",
            "roa"
        ]
    ].isna().sum()
)

revenue          73
net_income       73
total_assets     73
profit_margin    73
roa              73
dtype: int64


In [14]:
data.to_csv(
    "../datasets/processed/day14_financial_ratios_dataset.csv",
    index=False
)

print("Day 14 dataset saved successfully.")

Day 14 dataset saved successfully.


In [15]:
import pandas as pd
import numpy as np

In [16]:
data = pd.read_csv(
    "../datasets/processed/day14_financial_ratios_dataset.csv"
)

print("Shape:", data.shape)
print(data.columns.tolist())

Shape: (110, 19)
['company', 'ticker', 'year', 'year_end_price', 'dividend_per_share', 'dividend_yield', 'dividend_growth', 'dividend_growth_category', 'annual_stock_return', 'return_category', 'total_return', 'return_volatility', 'average_return', 'dividend_consistency', 'revenue', 'net_income', 'total_assets', 'profit_margin', 'roa']


In [17]:
print(
    data[
        [
            "revenue",
            "net_income",
            "total_assets",
            "profit_margin",
            "roa"
        ]
    ].describe()
)

            revenue    net_income  total_assets  profit_margin        roa
count  3.700000e+01  3.700000e+01  3.700000e+01      37.000000  37.000000
mean   2.013961e+11  3.200059e+10  2.444649e+11      19.703639  12.542455
std    1.848379e+11  3.279848e+10  1.548482e+11      10.421290   6.900867
min    2.318200e+10  6.177000e+09  5.043560e+10       1.910717   3.054266
25%    8.200600e+10  1.063100e+10  1.005490e+11      10.427417   7.672325
50%    9.419300e+10  1.597400e+10  1.992100e+11      18.952589  10.965886
75%    3.346970e+11  3.515300e+10  3.763170e+11      26.915064  15.083620
max    6.809850e+11  1.120100e+11  6.190030e+11      41.279254  31.179626


In [19]:
from sklearn.preprocessing import MinMaxScaler

normalization_columns = [
    "dividend_yield",
    "dividend_growth",
    "annual_stock_return",
    "total_return",
    "return_volatility",
    "profit_margin",
    "roa"
]

available_columns = [
    col for col in normalization_columns
    if col in data.columns
]

scaler = MinMaxScaler()

for col in available_columns:
    data[col + "_normalized"] = scaler.fit_transform(
        data[[col]]
    )

print("Normalized columns:")
print([col for col in data.columns if "_normalized" in col])

Normalized columns:
['dividend_yield_normalized', 'dividend_growth_normalized', 'annual_stock_return_normalized', 'total_return_normalized', 'return_volatility_normalized', 'profit_margin_normalized', 'roa_normalized']


In [20]:
data[
    [col for col in data.columns if "_normalized" in col]
].head()

,dividend_yield_normalized,dividend_growth_normalized,annual_stock_return_normalized,total_return_normalized,return_volatility_normalized,profit_margin_normalized,roa_normalized
0,0.192308,NaN,NaN,NaN,1.0,NaN,NaN
1,0.192308,0.666667,0.400976,0.371376,1.0,NaN,NaN
2,0.132754,0.698745,0.684869,0.668811,1.0,NaN,NaN
3,0.174938,1.000000,0.268629,0.229677,1.0,NaN,NaN
4,0.081886,0.523710,1.000000,1.000000,1.0,NaN,NaN


In [21]:
data.to_csv(
    "../datasets/processed/day9_normalized_dataset.csv",
    index=False
)

print("Day 9 normalization completed.")

Day 9 normalization completed.


In [22]:
print([col for col in data.columns if "_normalized" in col])

['dividend_yield_normalized', 'dividend_growth_normalized', 'annual_stock_return_normalized', 'total_return_normalized', 'return_volatility_normalized', 'profit_margin_normalized', 'roa_normalized']


In [23]:
data.to_csv(
    "../datasets/processed/day9_normalized_dataset.csv",
    index=False
)

print("Day 9 normalization completed.")

Day 9 normalization completed.


In [24]:
print("EPS exists:", "eps" in data.columns)
print("Available columns:")
print(data.columns.tolist())

EPS exists: False
Available columns:
['company', 'ticker', 'year', 'year_end_price', 'dividend_per_share', 'dividend_yield', 'dividend_growth', 'dividend_growth_category', 'annual_stock_return', 'return_category', 'total_return', 'return_volatility', 'average_return', 'dividend_consistency', 'revenue', 'net_income', 'total_assets', 'profit_margin', 'roa', 'dividend_yield_normalized', 'dividend_growth_normalized', 'annual_stock_return_normalized', 'total_return_normalized', 'return_volatility_normalized', 'profit_margin_normalized', 'roa_normalized']


In [25]:
print(
    data[
        ["ticker", "year", "dividend_per_share"]
    ].head(20).to_string(index=False)
)

ticker  year  dividend_per_share
  AAPL  2015              0.5075
  AAPL  2016              0.5575
  AAPL  2017              0.6150
  AAPL  2018              0.7050
  AAPL  2019              0.7600
  AAPL  2020              0.8075
  AAPL  2021              0.8650
  AAPL  2022              0.9100
  AAPL  2023              0.9500
  AAPL  2024              0.9900
  AAPL  2025              1.0300
   JNJ  2015              2.9500
   JNJ  2016              3.1500
   JNJ  2017              3.3200
   JNJ  2018              3.5400
   JNJ  2019              3.7500
   JNJ  2020              3.9800
   JNJ  2021              4.1900
   JNJ  2022              4.4500
   JNJ  2023              4.7000


In [26]:
print("EPS exists:", "eps" in data.columns)

EPS exists: False


In [27]:
print(data.columns.tolist())

['company', 'ticker', 'year', 'year_end_price', 'dividend_per_share', 'dividend_yield', 'dividend_growth', 'dividend_growth_category', 'annual_stock_return', 'return_category', 'total_return', 'return_volatility', 'average_return', 'dividend_consistency', 'revenue', 'net_income', 'total_assets', 'profit_margin', 'roa', 'dividend_yield_normalized', 'dividend_growth_normalized', 'annual_stock_return_normalized', 'total_return_normalized', 'return_volatility_normalized', 'profit_margin_normalized', 'roa_normalized']


In [28]:
fundamentals = pd.read_csv(
    "../datasets/raw/fundamentals_real.csv"
)

print("Fundamentals columns:")
print(fundamentals.columns.tolist())

Fundamentals columns:
['ticker', 'year', 'revenue', 'net_income', 'total_assets']


In [29]:
print(fundamentals.head())

  ticker  year       revenue    net_income  total_assets
0   AAPL  2025  4.161610e+11  1.120100e+11  3.592410e+11
1   AAPL  2024  3.910350e+11  9.373600e+10  3.649800e+11
2   AAPL  2023  3.832850e+11  9.699500e+10  3.525830e+11
3   AAPL  2022  3.943280e+11  9.980300e+10  3.527550e+11
4   AAPL  2021           NaN           NaN           NaN


In [30]:
import yfinance as yf

print("yfinance version:", yf.__version__)

yfinance version: 1.6.0


In [32]:
import yfinance as yf
import pandas as pd
import numpy as np

ticker = "AAPL"

stock = yf.Ticker(ticker)

income = stock.get_income_stmt()

print("Income statement shape:", income.shape)
print("Available rows:")
print(income.index.tolist())

Income statement shape: (39, 5)
Available rows:
['TaxEffectOfUnusualItems', 'TaxRateForCalcs', 'NormalizedEBITDA', 'NetIncomeFromContinuingOperationNetMinorityInterest', 'ReconciledDepreciation', 'ReconciledCostOfRevenue', 'EBITDA', 'EBIT', 'NetInterestIncome', 'InterestExpense', 'InterestIncome', 'NormalizedIncome', 'NetIncomeFromContinuingAndDiscontinuedOperation', 'TotalExpenses', 'TotalOperatingIncomeAsReported', 'DilutedAverageShares', 'BasicAverageShares', 'DilutedEPS', 'BasicEPS', 'DilutedNIAvailtoComStockholders', 'NetIncomeCommonStockholders', 'NetIncome', 'NetIncomeIncludingNoncontrollingInterests', 'NetIncomeContinuousOperations', 'TaxProvision', 'PretaxIncome', 'OtherIncomeExpense', 'OtherNonOperatingIncomeExpenses', 'NetNonOperatingInterestIncomeExpense', 'InterestExpenseNonOperating', 'InterestIncomeNonOperating', 'OperatingIncome', 'OperatingExpense', 'ResearchAndDevelopment', 'SellingGeneralAndAdministration', 'GrossProfit', 'CostOfRevenue', 'TotalRevenue', 'OperatingRe

In [33]:
eps_rows = [
    row for row in income.index
    if "eps" in str(row).lower()
]

print("EPS-related rows:")
print(eps_rows)

EPS-related rows:
['DilutedEPS', 'BasicEPS']


In [34]:
print(income.loc["DilutedEPS"])

2025-09-30    7.46
2024-09-30    6.08
2023-09-30    6.13
2022-09-30    6.11
2021-09-30     NaN
Name: DilutedEPS, dtype: float64


In [35]:
import yfinance as yf
import pandas as pd
import numpy as np

eps_data = []

tickers = data["ticker"].unique()

for ticker in tickers:
    print("Collecting EPS:", ticker)

    stock = yf.Ticker(ticker)
    income = stock.get_income_stmt()

    if income is None or income.empty:
        continue

    if "DilutedEPS" not in income.index:
        print("  DilutedEPS not available")
        continue

    eps_series = income.loc["DilutedEPS"]

    for date, eps in eps_series.items():

        year = pd.to_datetime(date).year

        eps_data.append({
            "ticker": ticker,
            "year": year,
            "eps": eps
        })

eps_df = pd.DataFrame(eps_data)

print("\nEPS DATA:")
print(eps_df.head(20))

print("\nEPS shape:", eps_df.shape)


EPS DATA:
   ticker  year    eps
0    AAPL  2025   7.46
1    AAPL  2024   6.08
2    AAPL  2023   6.13
3    AAPL  2022   6.11
4    AAPL  2021    NaN
5     JNJ  2025  11.03
6     JNJ  2024   5.79
7     JNJ  2023  13.72
8     JNJ  2022   6.73
9     JNJ  2021    NaN
10     KO  2025   3.04
11     KO  2024   2.46
12     KO  2023   2.47
13     KO  2022   2.19
14     KO  2021    NaN
15    MCD  2025  11.95
16    MCD  2024  11.39
17    MCD  2023  11.56
18    MCD  2022   8.33
19   MSFT  2026  17.95

EPS shape: (46, 3)


In [36]:
print(eps_df.groupby("ticker")["year"].apply(list))

ticker
AAPL    [2025, 2024, 2023, 2022, 2021]
JNJ     [2025, 2024, 2023, 2022, 2021]
KO      [2025, 2024, 2023, 2022, 2021]
MCD           [2025, 2024, 2023, 2022]
MSFT          [2026, 2025, 2024, 2023]
PEP     [2025, 2024, 2023, 2022, 2021]
PG            [2026, 2025, 2024, 2023]
VZ            [2025, 2024, 2023, 2022]
WMT     [2026, 2025, 2024, 2023, 2022]
XOM     [2025, 2024, 2023, 2022, 2021]
Name: year, dtype: object


In [37]:
data = data.merge(
    eps_df,
    on=["ticker", "year"],
    how="left"
)

print("New shape:", data.shape)
print(data[["ticker", "year", "dividend_per_share", "eps"]].head(20))

New shape: (110, 27)
   ticker  year  dividend_per_share    eps
0    AAPL  2015              0.5075    NaN
1    AAPL  2016              0.5575    NaN
2    AAPL  2017              0.6150    NaN
3    AAPL  2018              0.7050    NaN
4    AAPL  2019              0.7600    NaN
5    AAPL  2020              0.8075    NaN
6    AAPL  2021              0.8650    NaN
7    AAPL  2022              0.9100   6.11
8    AAPL  2023              0.9500   6.13
9    AAPL  2024              0.9900   6.08
10   AAPL  2025              1.0300   7.46
11    JNJ  2015              2.9500    NaN
12    JNJ  2016              3.1500    NaN
13    JNJ  2017              3.3200    NaN
14    JNJ  2018              3.5400    NaN
15    JNJ  2019              3.7500    NaN
16    JNJ  2020              3.9800    NaN
17    JNJ  2021              4.1900    NaN
18    JNJ  2022              4.4500   6.73
19    JNJ  2023              4.7000  13.72


In [38]:
data["payout_ratio"] = (
    data["dividend_per_share"] / data["eps"]
) * 100

In [39]:
print(
    data[
        ["ticker", "year", "dividend_per_share", "eps", "payout_ratio"]
    ].head(20).to_string(index=False)
)

ticker  year  dividend_per_share   eps  payout_ratio
  AAPL  2015              0.5075   NaN           NaN
  AAPL  2016              0.5575   NaN           NaN
  AAPL  2017              0.6150   NaN           NaN
  AAPL  2018              0.7050   NaN           NaN
  AAPL  2019              0.7600   NaN           NaN
  AAPL  2020              0.8075   NaN           NaN
  AAPL  2021              0.8650   NaN           NaN
  AAPL  2022              0.9100  6.11     14.893617
  AAPL  2023              0.9500  6.13     15.497553
  AAPL  2024              0.9900  6.08     16.282895
  AAPL  2025              1.0300  7.46     13.806971
   JNJ  2015              2.9500   NaN           NaN
   JNJ  2016              3.1500   NaN           NaN
   JNJ  2017              3.3200   NaN           NaN
   JNJ  2018              3.5400   NaN           NaN
   JNJ  2019              3.7500   NaN           NaN
   JNJ  2020              3.9800   NaN           NaN
   JNJ  2021              4.1900   NaN        

In [40]:
print(
    "Missing payout ratios:",
    data["payout_ratio"].isna().sum()
)

Missing payout ratios: 73


In [41]:
data.to_csv(
    "../datasets/processed/day11_payout_ratio_dataset.csv",
    index=False
)

print("Day 11 payout ratio dataset saved successfully.")

Day 11 payout ratio dataset saved successfully.


In [42]:
print(data.shape)
print(data.columns.tolist())

(110, 28)
['company', 'ticker', 'year', 'year_end_price', 'dividend_per_share', 'dividend_yield', 'dividend_growth', 'dividend_growth_category', 'annual_stock_return', 'return_category', 'total_return', 'return_volatility', 'average_return', 'dividend_consistency', 'revenue', 'net_income', 'total_assets', 'profit_margin', 'roa', 'dividend_yield_normalized', 'dividend_growth_normalized', 'annual_stock_return_normalized', 'total_return_normalized', 'return_volatility_normalized', 'profit_margin_normalized', 'roa_normalized', 'eps', 'payout_ratio']


In [43]:
print(
    data[
        ["ticker", "year", "dividend_per_share", "eps", "payout_ratio"]
    ].dropna(subset=["payout_ratio"]).head(30).to_string(index=False)
)

ticker  year  dividend_per_share   eps  payout_ratio
  AAPL  2022               0.910  6.11     14.893617
  AAPL  2023               0.950  6.13     15.497553
  AAPL  2024               0.990  6.08     16.282895
  AAPL  2025               1.030  7.46     13.806971
   JNJ  2022               4.450  6.73     66.121842
   JNJ  2023               4.700 13.72     34.256560
   JNJ  2024               4.910  5.79     84.801382
   JNJ  2025               5.140 11.03     46.600181
    KO  2022               1.760  2.19     80.365297
    KO  2023               1.840  2.47     74.493927
    KO  2024               1.940  2.46     78.861789
    KO  2025               2.040  3.04     67.105263
   MCD  2022               5.660  8.33     67.947179
   MCD  2023               6.230 11.56     53.892734
   MCD  2024               6.780 11.39     59.525900
   MCD  2025               7.170 11.95     60.000000
  MSFT  2023               2.790  9.68     28.822314
  MSFT  2024               3.080 11.80     26.

In [44]:
print("Payout ratio available:", data["payout_ratio"].notna().sum())
print("Payout ratio missing:", data["payout_ratio"].isna().sum())

Payout ratio available: 37
Payout ratio missing: 73


In [45]:
data["pe_ratio"] = (
    data["year_end_price"] / data["eps"]
)

In [46]:
print(
    data[
        ["ticker", "year", "year_end_price", "eps", "pe_ratio"]
    ].dropna(subset=["pe_ratio"]).head(30).to_string(index=False)
)

ticker  year  year_end_price   eps  pe_ratio
  AAPL  2022      129.929993  6.11 21.265138
  AAPL  2023      192.529999  6.13 31.407830
  AAPL  2024      250.419998  6.08 41.187500
  AAPL  2025      271.859985  7.46 36.442357
   JNJ  2022      176.649994  6.73 26.248142
   JNJ  2023      156.740005 13.72 11.424199
   JNJ  2024      144.619995  5.79 24.977547
   JNJ  2025      206.949997 11.03 18.762466
    KO  2022       63.610001  2.19 29.045662
    KO  2023       58.930000  2.47 23.858300
    KO  2024       62.259998  2.46 25.308942
    KO  2025       69.910004  3.04 22.996712
   MCD  2022      263.529999  8.33 31.636254
   MCD  2023      296.510010 11.56 25.649655
   MCD  2024      289.890015 11.39 25.451274
   MCD  2025      305.630005 11.95 25.575733
  MSFT  2023      376.040009  9.68 38.847108
  MSFT  2024      421.500000 11.80 35.720339
  MSFT  2025      483.619995 13.64 35.456011
   PEP  2022      180.660004  6.42 28.140187
   PEP  2023      169.839996  6.56 25.890243
   PEP  20

In [47]:
print("P/E available:", data["pe_ratio"].notna().sum())
print("P/E missing:", data["pe_ratio"].isna().sum())

P/E available: 37
P/E missing: 73


In [48]:
data.to_csv(
    "../datasets/processed/day12_pe_ratio_dataset.csv",
    index=False
)

print("Day 12 P/E ratio dataset saved successfully.")

Day 12 P/E ratio dataset saved successfully.


In [49]:
print(data.shape)
print("P/E column exists:", "pe_ratio" in data.columns)

(110, 29)
P/E column exists: True


In [50]:
data = data.sort_values(["ticker", "year"])

data["earnings_growth"] = (
    data.groupby("ticker")["eps"]
    .pct_change() * 100
)

In [51]:
print(
    data[
        ["ticker", "year", "eps", "earnings_growth"]
    ].dropna(subset=["earnings_growth"]).head(30).to_string(index=False)
)

ticker  year       eps  earnings_growth
  AAPL  2023  6.130000         0.327332
  AAPL  2024  6.080000        -0.815661
  AAPL  2025  7.460000        22.697368
   JNJ  2023 13.720000       103.863299
   JNJ  2024  5.790000       -57.798834
   JNJ  2025 11.030000        90.500864
    KO  2023  2.470000        12.785388
    KO  2024  2.460000        -0.404858
    KO  2025  3.040000        23.577236
   MCD  2023 11.560000        38.775510
   MCD  2024 11.390000        -1.470588
   MCD  2025 11.950000         4.916594
  MSFT  2024 11.800000        21.900826
  MSFT  2025 13.640000        15.593220
   PEP  2023  6.560000         2.180685
   PEP  2024  6.950000         5.945122
   PEP  2025  6.000000       -13.669065
    PG  2024  6.020000         2.033898
    PG  2025  6.510000         8.139535
    VZ  2023  2.750000       -45.652174
    VZ  2024  4.140000        50.545455
    VZ  2025  4.060000        -1.932367
   WMT  2024  1.913333        34.426238
   WMT  2025  2.410000        25.958210


In [52]:
print("Earnings growth available:",
      data["earnings_growth"].notna().sum())

print("Earnings growth missing:",
      data["earnings_growth"].isna().sum())

Earnings growth available: 27
Earnings growth missing: 83


In [53]:
data.to_csv(
    "../datasets/processed/day13_earnings_growth_dataset.csv",
    index=False
)

print("Day 13 earnings growth dataset saved successfully.")

Day 13 earnings growth dataset saved successfully.


In [54]:
print(data.shape)
print("Earnings growth column exists:",
      "earnings_growth" in data.columns)

(110, 30)
Earnings growth column exists: True


In [55]:
engineered_features = [
    "dividend_yield",
    "payout_ratio",
    "pe_ratio",
    "earnings_growth"
]

print("Engineered features:")
for feature in engineered_features:
    print(
        feature,
        "->",
        feature in data.columns
    )

Engineered features:
dividend_yield -> True
payout_ratio -> True
pe_ratio -> True
earnings_growth -> True


In [56]:
print("Final dataset shape:", data.shape)
print("\nFinal columns:")
print(data.columns.tolist())

Final dataset shape: (110, 30)

Final columns:
['company', 'ticker', 'year', 'year_end_price', 'dividend_per_share', 'dividend_yield', 'dividend_growth', 'dividend_growth_category', 'annual_stock_return', 'return_category', 'total_return', 'return_volatility', 'average_return', 'dividend_consistency', 'revenue', 'net_income', 'total_assets', 'profit_margin', 'roa', 'dividend_yield_normalized', 'dividend_growth_normalized', 'annual_stock_return_normalized', 'total_return_normalized', 'return_volatility_normalized', 'profit_margin_normalized', 'roa_normalized', 'eps', 'payout_ratio', 'pe_ratio', 'earnings_growth']


In [57]:
data.to_csv(
    "../datasets/processed/final_dividend_puzzle_dataset.csv",
    index=False
)

print("Final Dividend Puzzle dataset saved successfully.")

Final Dividend Puzzle dataset saved successfully.


In [58]:
import os

file_path = "../datasets/processed/final_dividend_puzzle_dataset.csv"

print("File exists:", os.path.exists(file_path))

if os.path.exists(file_path):
    print("File size:", os.path.getsize(file_path), "bytes")

File exists: True
File size: 28814 bytes


In [59]:
print("Shape:", data.shape)

print("\nMissing values:")
print(
    data[
        [
            "dividend_yield",
            "payout_ratio",
            "pe_ratio",
            "earnings_growth"
        ]
    ].isna().sum()
)

Shape: (110, 30)

Missing values:
dividend_yield      0
payout_ratio       73
pe_ratio           73
earnings_growth    83
dtype: int64
